# 03 · Big Data processing pipeline (Spark)

Builds the analysis tables from the cleaned inputs with Spark DataFrame operations:

| Step | Spark operations | Output | Used by |
|---|---|---|---|
| 1 | aggregation of 144 M speed-bin rows with window functions, broadcast join | sign-hour speed metrics | RQ2 |
| 2 | grouping + join with the full weather grid | city-hour panel (105,192 rows) | RQ1 frequency |
| 3 | grid-based spatial join (cell join + distance filter + window) | collision–sign matches (100/150/250/500 m) | RQ2 |
| 4 | filtering, grouping, distinct counts, joins | sign-level table | RQ2 |
| 5 | union of differently formatted files, timestamp parsing, broadcast join, grouping | neighbourhood-hour panel | RQ3 |

The statistical models are fitted afterwards on these much smaller tables (notebooks 04–06).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from pyspark.sql import functions as F
from src import panels as P, wys, spatial, bikeshare as B
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "|", spark.sparkContext.master)
coll = pd.read_parquet(PROC / "collisions.parquet")
coll = coll[coll["year"].between(2014, 2025)]
wx = pd.read_parquet(PROC / "weather_hourly.parquet")
locations = pd.read_parquet(PROC / "wys_locations.parquet")
weather_sdf = P.to_spark(spark, wx[P.WEATHER_KEEP]).cache()
coll_sdf = P.to_spark(spark, coll[["collision_id", "date", "hour", "severe", "bicycle", "pedestrian", "hood_158"]]).cache()
loc_sdf = P.to_spark(spark, locations[["sign_id", "speed_limit", "start_date", "end_date"]])

## Step 1 · WYS speed bins → sign-hour speed metrics

In [ ]:
wys_raw = spark.read.parquet(str(PROC / "wys_raw"))
print(f"input rows: {wys_raw.count():,}")
wys_raw.show(5, truncate=False)
sign_hour = wys.sign_hour_metrics(wys_raw.drop("year", "month"), loc_sdf)
sign_hour.write.mode("overwrite").parquet(str(PROC / "sign_hour"))
sign_hour = spark.read.parquet(str(PROC / "sign_hour"))
print(f"sign-hours: {sign_hour.count():,}")
sign_hour.orderBy("sign_id", "date", "hour").show(5)

In [ ]:
sh_panel = P.sign_hour_panel(sign_hour, weather_sdf, loc_sdf, cfg["wys"]["min_volume_per_hour"])
sh_panel.write.mode("overwrite").parquet(str(PROC / "sign_hour_panel"))
print(f"sign-hours with >= {cfg['wys']['min_volume_per_hour']} vehicles in the active period: {spark.read.parquet(str(PROC / 'sign_hour_panel')).count():,}")

## Step 2 · City-hour panel (grouping + join)

In [ ]:
city_hour = P.city_hour_panel(spark, coll, wx)
city_hour.select("date", "hour", "collisions", "severe_collisions", "rain", "snow", "temp_c", "visibility_km").show(5)
ch = city_hour.toPandas()
ch.to_parquet(PROC / "city_hour.parquet", index=False)
ch_nodup = P.city_hour_panel(spark, coll, wx, drop_duplicates=True).toPandas()
ch_nodup.to_parquet(PROC / "city_hour_nodup.parquet", index=False)
print(len(ch), "city-hours;", int(ch["collisions"].sum()), "collisions")
P.collision_level_table(coll, wx).to_parquet(PROC / "collision_level.parquet", index=False)

## Step 3 · Spatial join: collisions → nearest active WYS sign

In [ ]:
print("distance from each sign to the nearest collision point (m):")
display(spatial.nearest_distance_summary(coll, locations).round(1))
radii = sorted(set(cfg["spatial"]["e2_buffers_m"] + cfg["spatial"]["sensitivity_buffers_m"] + [cfg["spatial"]["buffer_m"]]))
match_counts = {}
for r in radii:
    m = spatial.match_collisions_to_signs(spark, coll, locations, r)
    m.write.mode("overwrite").parquet(str(PROC / f"matches_{r}m"))
    match_counts[r] = spark.read.parquet(str(PROC / f"matches_{r}m")).count()
pd.Series(match_counts, name="collisions matched to a sign").rename_axis("buffer (m)")

## Step 4 · Sign-level table (speeds, active days, nearby collisions)

In [ ]:
for r in radii:
    m = spark.read.parquet(str(PROC / f"matches_{r}m"))
    st = P.sign_level_table(sign_hour, m, coll_sdf, cfg["speed_years"]).toPandas()
    st.to_parquet(PROC / f"sign_level_{r}m.parquet", index=False)
st150 = pd.read_parquet(PROC / f"sign_level_{cfg['spatial']['buffer_m']}m.parquet")
st150.describe().round(2)

## Step 5 · Bike Share trips → neighbourhood-hour panel

In [ ]:
stations = B.stations_with_neighbourhood(cfg)
print(len(stations), "stations;", stations["hood_158"].nunique(), "neighbourhoods with stations")
trips = B.load_trips(spark, cfg)
trip_counts, match = B.neighbourhood_hour_trips(trips, P.to_spark(spark, stations[["station_id", "hood_158"]]))
display(match.orderBy("year").toPandas().round(3))     # share of trips whose start station is located
trip_counts.write.mode("overwrite").parquet(str(PROC / "trips_neighbourhood_hour"))
trip_counts = spark.read.parquet(str(PROC / "trips_neighbourhood_hour"))
nh = P.neighbourhood_hour_panel(trip_counts, coll_sdf, weather_sdf)
nh.write.mode("overwrite").parquet(str(PROC / "neighbourhood_hour"))
nh = spark.read.parquet(str(PROC / "neighbourhood_hour"))
nh.select("hood_158", "date", "hour", "trips", "cyclist_collisions", "rain", "snow").show(5)
print(f"neighbourhood-hours with trips: {nh.count():,}")

In [ ]:
# Row counts of every analysis table (for the report)
sizes = {"WYS speed-bin rows (2023-2025)": wys_raw.count(), "sign-hours": sign_hour.count(),
         "sign-hour panel": spark.read.parquet(str(PROC / "sign_hour_panel")).count(),
         "city-hours": len(ch), "collision-level rows": len(pd.read_parquet(PROC / "collision_level.parquet")),
         "neighbourhood-hours": nh.count()}
pd.Series(sizes).map("{:,}".format).to_frame("rows").to_csv(TAB / "pipeline_table_sizes.csv")
pd.Series(sizes).map("{:,}".format).to_frame("rows")

In [ ]:
spark.stop()